# Final Report: Porting C and C++ Courses to xeus-cpp

---


## 1. Motivation and Problem Statement

Teaching C and C++ traditionally involves a friction-heavy loop: write code in a text editor, switch to a terminal, compile, fix errors, recompile, and only then see output. For beginners, this cycle interrupts the learning flow before they have had a chance to build intuition.

Interactive notebook environments have transformed the teaching of Python and data science by collapsing this cycle - write a line, see the result immediately, iterate. xeus-cpp brings this interactivity to C and C++ through a Clang/LLVM JIT backend that evaluates C++ code cell by cell, preserving state between executions.

However, xeus-cpp's execution model is fundamentally different from traditional compilation in ways that are not obvious to either students or instructors. A naive port of existing course material fails because patterns that work in a compiled program - defining variables in multiple places, relying on `int main()`, expecting fresh state on each run - do not behave the same way in a notebook. This project addressed that gap directly.

---

## 2. Deliverables



| Notebook | Week | Topics Covered |
|---|---|---|
| `01_verification` | 1 | Kernel verification, state persistence, execution model comparison |
| `02_persistence` | 2 | Explaining xeus-cpp state persistence
| `03_redefintition` | 3 | Redefinition conflicts, safe coding patterns |
| `04_c_fundamentals` | 4 | Types, operators, control flow, functions, arrays, structs |
| `05_pointers_and_memory` | 5 | Pointer arithmetic, dynamic memory, dangling pointers, memory rules |
| `06_cpp_oop` | 6 | Classes, constructors, operator overloading, inheritance, polymorphism |
| `07_stl_modern_cpp` | 7 | STL containers, algorithms, lambdas, smart pointers, optional, variant |
| `08_file_io_exceptions` | 8 | File I/O, string streams, custom exceptions, RAII |
| `09_advanced_templates_inheritance` | 9 | Templates, specialization, variadic templates, C++20 concepts |
| `10_modular_notebook_design` | 10 | Scalable notebook structure, multi-module design pattern |
| - | 11-12 | Buffer week - bug fixes, notebook refinement |
| `visual_guides_state_flow` | 13 | Execution model, memory allocation, workflow comparison |
| `project_final_report` | 14–15 | This document |



---

## 3. Technical Work

### 3.1 Understanding the xeus-cpp Execution Model

The most significant technical challenge was understanding how xeus-cpp's persistent state model affects C++ teaching. The key differences from traditional compilation are:

| Aspect | g++/clang++ | xeus-cpp |
|---|---|---|
| State | Resets on each compile | Persists until kernel restart |
| Execution unit | Entire program | Individual cells |
| Entry point | `int main()` required | Not needed — conflicts with kernel |
| Includes | Per translation unit | Persist once included |
| Variable redefinition | Compile error | May silently shadow or conflict |
| Class redefinition | Compile error | Error — requires `#ifndef` guard |
| Error scope | Compilation halts entirely | Only the failing cell halts |

This required rethinking how examples are structured. A program that works perfectly as a `.cpp` file can behave incorrectly in a notebook if cells are re-run out of order or if type definitions appear more than once.

### 3.2 The Safe Coding Patterns

Five patterns were developed and documented to let notebook authors work safely within the stateful model:

**Pattern 1 — Assignment over re-declaration.**  
Once a variable exists in state, updating it requires plain assignment (`x = 10`) rather than re-declaration (`int x = 10`). Re-declaration either errors or silently resets the value.

**Pattern 2 — Lambdas for function iteration.**  
When testing a revised version of a function, using a lambda in a block scope avoids creating a conflicting global definition.

**Pattern 3 — Namespace versioning.**  
Wrapping different versions of the same function in `namespace v1`, `namespace v2` allows both to coexist across cells.

**Pattern 4 — `#ifndef` include guards for class definitions.**  
Re-running a cell that defines a class triggers a redefinition error. Wrapping the definition in a preprocessor guard prevents this entirely.

**Pattern 5 — Recursion depth guards.**  
Deep recursion crashes the kernel with a stack overflow. Adding a `depth` parameter with a hard ceiling prevents this and gives a clean error instead.

### 3.3 The Modular Notebook Structure

A consistent section template was designed and applied across all notebooks:

```
1. Title and learning objectives
2. Setup cell (all #includes, using declarations)
3. Concept introduction          
4. Minimal working example      
5. Extended example              
6. Edge cases and common mistakes 
7. xeus-cpp specific notes        

```

This structure matters in a stateful environment because it enforces a single, correct execution order. A student who follows the structure top to bottom will not encounter state-related bugs.


---

## 4. Timeline and Progress

| Week | Planned | Completed |
|---|---|---|
| 1–2 | Environment setup, understand xeus-cpp model | Setup notebook + execution model documentation |
| 3 | Investigate redefinition and state issues | Safe coding pattern catalog (5 patterns) |
| 4 | Port C fundamentals | Notebook covering types, control flow, functions, structs |
| 5 | Port C memory management | Notebook covering pointers, malloc/free, dangling pointers |
| 6 | Port C++ OOP | Notebook covering classes, inheritance, polymorphism |
| 7 | Port STL and modern C++ | Notebook covering STL, lambdas, smart pointers, C++17 features |
| 8 | Port file I/O and exceptions | Notebook covering streams, RAII, custom exceptions |
| 9 | Port advanced C++ | Notebook covering templates, concepts, variadic templates |
| 10 | Design scalable notebook structure | Modular pattern notebook with full TaskManager example |
| 11-12 | Debug and Refine | Cleaned-up syntax and markdown |
| 13 | Visual guides and diagrams | Execution model, memory layout |
| 14–15 | Final report and submission | This document + all supporting docs |

---

## 5. Lessons Learned

### 5.1 What Worked Well

Building examples one cell at a time made it easy to isolate bugs and keep each concept focused. It also forced a natural structure that benefits students reading the notebook later.

Applying the same structure to every notebook made them consistent and predictable. Once a student learns to read one notebook, they can navigate all of them without re-orienting.

Visual explanations of the execution model and memory layout added significant pedagogical value.

### 5.2 Challenges

The fact that re-running cells out of order produces different results than running them in order is unintuitive, especially for students who have only ever used compiled languages. It required dedicated notebook space and explicit warnings throughout.

Students learning C++ from books or courses are told `int main()` is mandatory. In xeus-cpp it is forbidden. This required clear, repeated callouts to prevent confusion.

When a template constraint is violated, Clang produces error messages that span many lines. For a beginner reading a notebook, this can be overwhelming. The partial solution — using C++20 concepts to produce targeted errors — works well but requires the student to already understand concepts.

C and C++ together span a very large curriculum. The original scope had to be carefully prioritized to ensure depth over breadth — each notebook needed enough examples and exercises to be useful as a standalone teaching resource, not just a code reference.
